# 04 — Mobility × Ticket 車輛配對可識別性：最終稽核報告

> 結論：**3. Matching requires additional identifiers/information（目前若要做可信的實體車輛 crosswalk，仍需要額外識別資訊）**。

`route × time` 確實含有訊號，也能找出月尺度上看似很強的候選；但日別最佳候選不穩、同分候選多、GPS top-1 碰撞多。現階段可把候選排名拿去人工或外部資料驗證，不能直接當最終實體車輛對照表。所有結果皆由前三本 notebook 直接讀 `data/` 原始日檔計算，沒有輸出 cleaned、trimmed、cache、CSV 或 Parquet。

## A. Dataset integrity

- Mobility：19 檔、03-11 至 03-31（缺 03-18、03-19），共 **13,850,467** 列。Ticket：31 檔、整個 3 月，共 **6,790,681** 列。
- 指定 matching 欄位缺失率皆為 0%；兩側沒有壞時間或檔名日期錯置。Mobility 完全重複列 0；Ticket **7,734** 列（0.114%）完全重複，只計數、未刪除。
- 已知壞日 03-18/19/20/22/28 不進主要配對。另發現 **03-17 在 10:46:51 結束**；正式研究應再做排除 03-17 的 sensitivity check。

## B. Vehicle ID structure

- Ticket `vehicle_number` **556** 個；Mobility `id` **527** 個；精確交集 **0**，數值範圍也分離（2,001–24,142 vs 52,243–136,462）。
- **23 個 `vehicle_number` 跨兩家公司重用**，故 Ticket 車鍵必須是 **`(company_number, vehicle_number)`**；可靠重疊期可評估 569 個複合車鍵。
- Ticket 每車觀測日中位數 25；Mobility 每 ID 中位數 13。大量短存續 GPS ID 與硬體更換相容，但本資料無法證明。

## C. Route compatibility

- 保守 normalization 後 Ticket **58** 條、Mobility **51** 條，交集 **46**。**92.75% Ticket 交易**及 **96.46% Mobility 觀測**的路線碼存在另一側，故 `route_name ↔ lineId` 可用但不完整。
- 只把整數型 `.0` 視為格式差異；`026`、`03`、`24.A`、`49.1` 等保留。
- `view_type ↔ lineName` 精確交集 **0**，只做大小寫/空白/去音調後也僅 1 個相同；不能靠 fuzzy name 強迫配對。
- `route_detail_id` 多半是 Ticket 內的 route/company 分類。close joins 中多數 detail 對應多個 `tripId`、兩個方向及多個 headsign，故 **不等於 `tripId`，通常也不是穩定方向 ID**。

## D. Temporal compatibility

- 照原標示直接對 BRT 牆鐘時，日內形狀不合。路線×小時 log-volume 相關：0 分鐘 **0.448**、+60 為 0.538、**+180 為 0.770**。
- 同路線零候選車分鐘由 0 分鐘 **42.91%** 降至 +180 的 **36.74%**；平均候選數由 5.57 增至 6.98；代表日縱向 evidence 亦在 +180 最高。
- 主診斷因此使用 **Ticket timestamp +3 小時**。這是實證 clock correction；在資料提供者確認前仍是不確定性。

## E. Single-time ambiguity

- 未修正時間的同路線 ±1 分鐘候選數：中位 **2**、平均 **5.57**、p90 **15**、最大 **39**；42.91% 為 0 候選，僅 **5.26%** 恰好 1 候選。單一 boarding 無法辨識實體車。
- 同一 GPS `id × minute` 多路線有 **990** 例（0.030%）；30 秒格 668 例，均保留並報告。

## F. Longitudinal matching strength

- ±30/60/90/120 秒的最佳 compatibility 中位數皆 1.0、最佳 `N_same` 中位約 1,385–1,419；但大量候選同為 1.0，因為車長期跑同一路線或缺測很多。
- 569 車中，保守門檻（compatibility≥0.98、second gap≥0.10、`N_same`≥100、≥3 天）僅 **105** 車；再要求 reciprocal top-1 剩 **90 對（15.8%）**。這仍只是 diagnostic subset。
- clear 比例隨窗長由 4 小時 **0.83%**、1 日 **2.39%**、3 日 **5.34%**，到整個可靠期 **23.37%**；整期 median second-best gap 仍僅 **0.033**。

## G. Day-to-day stability

- 90 對 preliminary high-confidence reciprocal 候選的 consistency 中位數僅 **0.50**；只有 **5/90（5.56%）** 每個支持日都選同一 ID，**13/90（14.44%）** consistency≥0.8。其餘 479 車中位數 0.304。
- 這是不能直接建立月度一對一 crosswalk 的最關鍵證據。跳換可能是 transponder replacement，也可能只是同路線車隊中互換；目前無法區分。

## H. Ambiguous or problematic vehicles

- `56::24142`：best `108650` (`N_same`=1,356, comp=1.0)，second `69181` (`N_same`=1,333, comp=1.0)，gap=0。
- `3::12025`：月 best `99006`，唯一支持日 best `99005`；`22::11027`：月 best `97136`，兩個支持日都選 `97094`；`2::13055`：月 best `99155`，日別選 `99113`/`99164`。
- `11::14001` 等無覆蓋車鍵所有候選 `N_same=0`，top-1 無配對意義。
- **41.30% Ticket 車鍵**的 independent top-1 與別車共用 GPS ID；`102866` 被 21 車列為 top-1、`72702` 被 9 車列為 top-1。不能用 greedy/Hungarian 強迫消除後當作證據。

## I. Event-level join feasibility

- 對 90 對 preliminary 候選做 +3h nearest join：**418,266** 筆有 10 分鐘內 GPS，**61,476** 筆無。時間差 median 4 秒、mean 20.0、p90 8、p95 118、p99 428 秒。
- 有 join 者 **91.93% ≤15s、92.51% ≤30s、93.40% ≤60s、95.06% ≤120s**；路線一致率 **99.36%**。
- 這表示若 crosswalk 已由外部證據確認，事件定位很有機會可靠；但同路線上選到另一輛車也會得到漂亮指標，不能反向證明車輛 identity。

## J. Recommendation

### **3. Matching requires additional identifiers/information**

**有機會 match，但目前只能產生候選，不能可靠定案。** 建議取得至少一種外部 anchor：車牌/車隊編號與 transponder 安裝表、車庫 dispatch/blocks、短期人工抽查，或 AVL/validator 共同事件。少量已知配對即可校驗 +3h、辨認 ID replacement，並量測 score 的真陽性率。

補資料前可保留 90 對作人工驗證 queue；若一定要做 sensitivity analysis，應再限縮到 consistency≥0.8 的約 13 對並標為 provisional，而非 crosswalk。不要對其餘車輛執行全域 Hungarian assignment。

## Reproducibility / cleaning decisions

ID 與 route 全以字串讀入；只 trim 外側空白並將純整數 `.0` 視為格式差異；沒有 fuzzy 合併。壞日只從 primary matching 排除，未刪原始資料。Ticket duplicates 只計數未刪。`missing` 與 `conflict` 分開，missing 不進 compatibility 分母。派生表只存在 notebook 記憶體與 cell output；資料夾只保留四本 `.ipynb`。